# Building the Qwen Microtext Corpus

This notebook reads our two annotation csv files and creates the final corpus json file that has the same format as the original Peldszus & Stede (2016) corpus.

We need:
- `manual_analysis_results.csv` (144 texts, has edu/adu segmentation and cleaned text)
- `adu_annotations.csv` (138 kept texts, has the pro/opp roles and argumentation relations)

In [2]:
import json
import csv
import re
from collections import Counter

## load the files

In [4]:
# main csv has all 144 texts including the discarded ones
main_data = {}
with open('../manual_analysis_results.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        main_data[int(row['text_id'])] = row

# adu file only has the 138 we kept
adu_data = {}
with open('../adu_annotations.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        adu_data[int(row['text_id'])] = row

print(f"main csv: {len(main_data)} texts")
print(f"adu csv:  {len(adu_data)} texts")

# these are the ones we threw out
discarded = sorted(set(main_data.keys()) - set(adu_data.keys()))
print(f"discarded: {discarded}")

main csv: 144 texts
adu csv:  138 texts
discarded: [96, 97, 98, 99, 100, 102]


## topic ids

these come from the topics_triggers.md file in the original corpus

In [5]:
topic_ids = {
    "Should Germany introduce the death penalty?": "introduce_capital_punishment",
    "Should intelligence services be regulated more tightly by parliament?": "stricter_regulation_of_intelligence_services",
    "Should the fine for leaving dog excrements on sideways be increased?": "higher_dog_poo_fines",
    "Should only those viewers pay a TV licence fee who actually want to watch programs offered by public broadcasters?": "public_broadcasting_fees_on_demand",
    "Should the morning-after pill be sold over the counter at the pharmacy?": "over_the_counter_morning_after_pill",
    "Should all universities in Germany charge tuition fees?": "charge_tuition_fees",
    "Should parts of the Tempelhofer Feld be made available for residential construction?": "partial_housing_development_at_Tempelhofer_Feld",
    "Should the weight of the BA thesis in the final grade be increased?": "increase_weight_of_BA_thesis_in_final_grade",
    "Should there be a cap on rent increases for a change of tenant?": "cap_rent_increases",
    "Should public health insurance cover treatments in complementary and alternative medicine?": "health_insurance_cover_complementary_medicine",
    "Should shopping malls generally be allowed to open on holidays and Sundays?": "allow_shops_to_open_on_holidays_and_sundays",
    "Should the EU exert influence on the political events in Ukraine?": "EU_influence_on_political_events_in_Ukraine",
    "Should the statutory retirement age remain at 63 years in the future?": "keep_retirement_at_63",
    "Should Germany buy CDs with tax evader data from dubious sources?": "buy_tax_evader_data_from_dubious_sources",
    "Should the Berlin Tegel airport remain operational after the opening of the Berlin Brandenburg airport?": "TXL_airport_remain_operational_after_BER_opening",
    "Should video games be made olympic?": "make_video_games_olympic",
    "Should school uniforms be worn again in our schools?": "school_uniforms",
    "Should we continue to separate our waste for recycling?": "waste_separation",
}

## seg mapping

the csv has a `seg_mapping` column that looks like `e1:a1, e2:a2, e3:a3, e10:a1`. easy to parse.

In [6]:
def get_seg(seg_string):
    # parse "e1:a1, e2:a2, e3:a3, e10:a1" into list of tuples
    pairs = []
    for pair in seg_string.split(', '):
        e, a = pair.split(':')
        pairs.append((e.strip(), a.strip()))
    return pairs

# test
print("text 0:", get_seg(main_data[0]['seg_mapping']))
print("text 26:", get_seg(main_data[26]['seg_mapping']))

text 0: [('e1', 'a1'), ('e2', 'a2'), ('e3', 'a3'), ('e4', 'a4'), ('e5', 'a5'), ('e6', 'a6'), ('e7', 'a7'), ('e8', 'a8'), ('e9', 'a9'), ('e10', 'a1')]
text 26: [('e1', 'a1'), ('e2', 'a2'), ('e3', 'a3'), ('e4', 'a4'), ('e5', 'a5'), ('e6', 'a2'), ('e7', 'a6')]


## extracting adu types and relations from the adu csv

the adu csv has columns like `adu_1_role` (pro or opp) and `adu_1_label` which is either `(0, root)` for the central claim or `(target, type)` like `(1, sup)` for a support relation to adu 1.

In [7]:
def get_adus(adu_row):
    types = []
    rels = []
    
    i = 1
    while adu_row.get(f'adu_{i}_role', ''):
        types.append(adu_row[f'adu_{i}_role'])
        
        label = adu_row[f'adu_{i}_label']
        if label != '(0, root)' and label:
            m = re.match(r'\((\d+),\s*(\w+)\)', label)
            if m:
                rels.append((i, int(m.group(1)), m.group(2)))
        i += 1
    
    return types, rels

# test
types, rels = get_adus(adu_data[0])
print(f"text 0: types={types}")
print(f"text 0: rels={rels}")

text 0: types=['pro', 'pro', 'pro', 'pro', 'pro', 'opp', 'pro', 'pro', 'pro']
text 0: rels=[(2, 1, 'sup'), (3, 1, 'sup'), (4, 1, 'sup'), (5, 4, 'sup'), (6, 1, 'reb'), (7, 6, 'reb'), (8, 1, 'sup'), (9, 1, 'sup')]


## build the corpus

In [8]:
corpus = []

for idx in sorted(adu_data.keys()):
    row = main_data[idx]
    adu_row = adu_data[idx]
    
    tid = topic_ids.get(row['topic'], row['topic'].lower().replace(' ', '_'))
    stance = "pro" if row['actual_stance'] == 'for' else "con"
    adu_count = int(row['adu_count'])
    
    # edus from the segmentation column
    edus = []
    for i, raw in enumerate(row['edu_segmentation'].split(' | ')):
        clean = re.sub(r'^\[\d+\]\s*', '', raw.strip())
        edus.append({"@id": f"e{i+1}", "#text": clean})
    
    # seg mapping from adu_content
    seg = get_seg(row['seg_mapping'])
    
    # adu types and relations from adu csv
    adu_types, rels = get_adus(adu_row)
    adus = [{"@id": f"a{i+1}", "@type": t} for i, t in enumerate(adu_types)]
    
    # put the edges together — relations first, then seg
    edges = []
    for i, (src, trg, rtype) in enumerate(rels):
        edges.append({"@id": f"c{i+1}", "@src": f"a{src}", "@trg": f"a{trg}", "@type": rtype})
    
    for i, (e_id, a_id) in enumerate(seg):
        edges.append({"@id": f"c{len(rels) + i + 1}", "@src": e_id, "@trg": a_id, "@type": "seg"})
    
    corpus.append({
        "arggraph": {
            "@id": f"micro_q{idx:03d}",
            "@topic_id": tid,
            "@stance": stance,
            "edu": edus,
            "adu": adus,
            "edge": edges
        },
        "@text": row['text_cleaned']
    })

print(f"done — {len(corpus)} texts")

done — 138 texts


## check that everything looks right

In [9]:
n_edus = sum(len(x['arggraph']['edu']) for x in corpus)
n_adus = sum(len(x['arggraph']['adu']) for x in corpus)
n_seg = sum(1 for x in corpus for e in x['arggraph']['edge'] if e['@type'] == 'seg')
n_rel = sum(1 for x in corpus for e in x['arggraph']['edge'] if e['@type'] != 'seg')

print(f"texts:      {len(corpus)}")
print(f"EDUs:       {n_edus}")
print(f"ADUs:       {n_adus}")
print(f"seg edges:  {n_seg} (should be {n_edus})")
print(f"rel edges:  {n_rel} (should be {n_adus - len(corpus)})")

# check for problems
bad_ids = sum(1 for x in corpus if len([e['@id'] for e in x['arggraph']['edge']]) != len(set(e['@id'] for e in x['arggraph']['edge'])))
bad_roots = sum(1 for x in corpus if len([a['@id'] for a in x['arggraph']['adu'] if a['@id'] not in set(e['@src'] for e in x['arggraph']['edge'] if e['@type'] != 'seg')]) != 1)
bad_seg = sum(1 for x in corpus if set(e['@id'] for e in x['arggraph']['edu']) != set(e['@src'] for e in x['arggraph']['edge'] if e['@type'] == 'seg'))

print(f"\nduplicate edge ids: {bad_ids}")
print(f"wrong number of roots: {bad_roots}")
print(f"seg coverage issues: {bad_seg}")

texts:      138
EDUs:       991
ADUs:       925
seg edges:  991 (should be 991)
rel edges:  787 (should be 787)

duplicate edge ids: 0
wrong number of roots: 0
seg coverage issues: 0


## save it

In [10]:
with open('qwen_microtext_corpus_ip.json', 'w', encoding='utf-8') as f:
    json.dump(corpus, f, indent=2, ensure_ascii=False)

print(f"saved ({len(corpus)} texts)")

saved (138 texts)


## look at some examples

In [11]:
# first text
print(json.dumps(corpus[0], indent=2, ensure_ascii=False)[:1000])

{
  "arggraph": {
    "@id": "micro_q000",
    "@topic_id": "introduce_capital_punishment",
    "@stance": "con",
    "edu": [
      {
        "@id": "e1",
        "#text": "Germany should not reintroduce the death penalty"
      },
      {
        "@id": "e2",
        "#text": "because it undermines human dignity"
      },
      {
        "@id": "e3",
        "#text": "and fails to deter crime effectively"
      },
      {
        "@id": "e4",
        "#text": "Capital punishment is inherently cruel and irreversible"
      },
      {
        "@id": "e5",
        "#text": "violating fundamental rights enshrined in international law"
      },
      {
        "@id": "e6",
        "#text": "Opponents argue that the death penalty serves as a necessary deterrent against heinous crimes"
      },
      {
        "@id": "e7",
        "#text": "but this overlooks the lack of conclusive evidence supporting its effectiveness compared to life imprisonment"
      },
      {
        "@id": "e8",
   

In [12]:
# find one with a restatement
for x in corpus:
    segs = [e for e in x['arggraph']['edge'] if e['@type'] == 'seg']
    trgs = [e['@trg'] for e in segs]
    if len(trgs) != len(set(trgs)):
        print(f"{x['arggraph']['@id']}:")
        print(f"  {len(x['arggraph']['edu'])} edus, {len(x['arggraph']['adu'])} adus")
        for e in segs:
            r = " ← restatement" if trgs.count(e['@trg']) > 1 else ""
            print(f"  {e['@src']} → {e['@trg']}{r}")
        break

micro_q000:
  10 edus, 9 adus
  e1 → a1 ← restatement
  e2 → a2
  e3 → a3
  e4 → a4
  e5 → a5
  e6 → a6
  e7 → a7
  e8 → a8
  e9 → a9
  e10 → a1 ← restatement


In [13]:
# overall numbers
rtypes = Counter(e['@type'] for x in corpus for e in x['arggraph']['edge'] if e['@type'] != 'seg')
atypes = Counter(a['@type'] for x in corpus for a in x['arggraph']['adu'])
rest = sum(1 for x in corpus if len(x['arggraph']['edu']) != len(x['arggraph']['adu']))

print(f"sup: {rtypes['sup']}, reb: {rtypes['reb']}")
print(f"pro: {atypes['pro']}, opp: {atypes['opp']}")
print(f"restatements: {rest}/{len(corpus)}")

sup: 437, reb: 350
pro: 750, opp: 175
restatements: 66/138
